In [2]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ImageNet stats for working in normalized space
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

def denormalize(x):
    mean = IMAGENET_MEAN.to(x.device).view(3,1,1)
    std  = IMAGENET_STD.to(x.device).view(3,1,1)
    return x * std + mean

def renormalize(x):
    mean = IMAGENET_MEAN.to(x.device).view(3,1,1)
    std  = IMAGENET_STD.to(x.device).view(3,1,1)
    return (x - mean) / std

def apply_gaussian_noise(x, sigma):
    # x is a batch, normalized
    x = denormalize(x)
    x = torch.clamp(x + torch.randn_like(x) * sigma, 0, 1)
    return renormalize(x)

def apply_motion_blur(x):
    # x is a batch [B, 3, H, W]
    kernel_size = 15
    kernel = torch.zeros((1, 1, kernel_size, kernel_size), device=x.device)
    kernel[:, :, kernel_size // 2, :] = 1.0 / kernel_size
    kernel = kernel.repeat(3, 1, 1, 1)
    x = denormalize(x)
    blurred = F.conv2d(x, kernel, padding=kernel_size//2, groups=3)
    return renormalize(torch.clamp(blurred, 0, 1))

def apply_brightness_shift(x, value=0.2):
    x = denormalize(x)
    x = torch.clamp(x + value, 0, 1)
    return renormalize(x)

def format_metrics(acc_corrupted, acc_clean):
    return {
        "Accuracy":            acc_corrupted,
        "Corruption Error":    1 - acc_corrupted,
        "Relative Robustness": acc_corrupted / acc_clean if acc_clean > 0 else 0
    }

def calculate_accuracy(model, loader, device, corruption=None, **kwargs):
    correct = 0
    total   = 0
    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images, labels = images.to(device), labels.to(device)

            if corruption == 'gaussian':
                images = apply_gaussian_noise(images, kwargs['sigma'])
            elif corruption == 'blur':
                images = apply_motion_blur(images)
            elif corruption == 'brightness':
                images = apply_brightness_shift(images)

            outputs   = model(images)
            _, preds  = torch.max(outputs, 1)
            total    += labels.size(0)
            correct  += (preds == labels).sum().item()
    return correct / total

@torch.no_grad()
def evaluate_robustness(model, dataloader, device):
    model.eval()
    results   = {}
    sigmas    = [0.05, 0.1, 0.2]

    clean_acc = calculate_accuracy(model, dataloader, device, corruption=None)
    print(f"Clean Accuracy: {clean_acc:.4f}")

    for s in sigmas:
        acc = calculate_accuracy(model, dataloader, device, corruption='gaussian', sigma=s)
        results[f'Gaussian_σ={s}'] = format_metrics(acc, clean_acc)

    blur_acc   = calculate_accuracy(model, dataloader, device, corruption='blur')
    results['Motion_Blur'] = format_metrics(blur_acc, clean_acc)

    bright_acc = calculate_accuracy(model, dataloader, device, corruption='brightness')
    results['Brightness_Shift'] = format_metrics(bright_acc, clean_acc)

    return results, clean_acc

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Dirs and device
train_dir = "/kaggle/input/datasets/mvpranay/aid-dataset/train_data"
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset split (same as 4.1)
train_full = datasets.ImageFolder(train_dir, transform=train_transform)
val_full   = datasets.ImageFolder(train_dir, transform=val_transform)
num_classes = len(train_full.classes)

train_size = int(0.7 * len(train_full))
val_size   = len(train_full) - train_size

torch.manual_seed(42)
train_dataset, _ = torch.utils.data.random_split(train_full, [train_size, val_size])
torch.manual_seed(42)
_, val_dataset   = torch.utils.data.random_split(val_full, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, persistent_workers=True)

# Model
model     = timm.create_model('resnet50', pretrained=True, num_classes=num_classes)
model     = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
epochs = 30
best_val_acc = 0.0

print(f"Training on {device} | {len(train_dataset)} train / {len(val_dataset)} val | {num_classes} classes")
print("="*60)

for epoch in range(epochs):
    # Train
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
    train_acc = correct / total

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            _, preds = torch.max(model(images), 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    val_acc = correct / total

    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "resnet50_aid.pth")
        print(f"  ✓ Saved best model (val_acc={val_acc:.4f})")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

Training on cuda | 4895 train / 2098 val | 30 classes


Epoch 01/30 | Loss: 3.1860 | Train: 0.2360 | Val: 0.4938
  ✓ Saved best model (val_acc=0.4938)


Epoch 02/30 | Loss: 2.0389 | Train: 0.6296 | Val: 0.7846
  ✓ Saved best model (val_acc=0.7846)


Epoch 03/30 | Loss: 0.8968 | Train: 0.8161 | Val: 0.8718
  ✓ Saved best model (val_acc=0.8718)


Epoch 04/30 | Loss: 0.4803 | Train: 0.8827 | Val: 0.9166
  ✓ Saved best model (val_acc=0.9166)


...

Epoch 26/30 | Loss: 0.0110 | Train: 0.9978 | Val: 0.9485


Epoch 27/30 | Loss: 0.0143 | Train: 0.9959 | Val: 0.9495


Epoch 28/30 | Loss: 0.0155 | Train: 0.9955 | Val: 0.9476


Epoch 29/30 | Loss: 0.0113 | Train: 0.9975 | Val: 0.9495


Epoch 30/30 | Loss: 0.0112 | Train: 0.9975 | Val: 0.9509

Training complete. Best val accuracy: 0.9528


In [13]:
import torch
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_dir   = "/kaggle/input/datasets/mvpranay/aid-dataset/train_data"
checkpoint = "/kaggle/input/datasets/mvpranay/gnr-models/resnet50_aid_test.pth"  # your saved weights

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
full_loader  = DataLoader(full_dataset, batch_size=32, shuffle=False, num_workers=4)
num_classes  = len(full_dataset.classes)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model with trained weights
model = timm.create_model('resnet50', pretrained=False, num_classes=num_classes)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model = model.to(device)

print(f"Evaluating on {len(full_dataset)} images across {num_classes} classes...")
results, clean_acc = evaluate_robustness(model, full_loader, device)

print("\n" + "="*65)
print(f"{'Corruption':<20} | {'Accuracy':<10} | {'Error':<10} | {'Rel. Robustness'}")
print("-"*65)
print(f"{'Clean':<20} | {clean_acc:.4f}     | {1-clean_acc:.4f}     | 1.0000")
for corruption, m in results.items():
    print(f"{corruption:<20} | {m['Accuracy']:.4f}     | {m['Corruption Error']:.4f}     | {m['Relative Robustness']:.4f}")
print("="*65)

Evaluating on 6993 images across 30 classes...


Clean Accuracy: 0.9834



Corruption           | Accuracy   | Error      | Rel. Robustness
-----------------------------------------------------------------
Clean                | 0.9834     | 0.0166     | 1.0000
Gaussian_σ=0.05      | 0.8872     | 0.1128     | 0.9021
Gaussian_σ=0.1       | 0.4527     | 0.5473     | 0.4604
Gaussian_σ=0.2       | 0.0997     | 0.9003     | 0.1014
Motion_Blur          | 0.5730     | 0.4270     | 0.5827
Brightness_Shift     | 0.9693     | 0.0307     | 0.9856


In [6]:
import torch
import timm
from torchvision import datasets, transforms, models
from torchvision.models import ConvNeXt_Tiny_Weights
from torch.utils.data import DataLoader
import torch.nn as nn

data_dir   = "/kaggle/input/datasets/mvpranay/aid-dataset/train_data"
checkpoint = "/kaggle/input/datasets/mvpranay/gnr-models/convnext_tiny_aid.pth"  # your saved weights

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
full_loader  = DataLoader(full_dataset, batch_size=32, shuffle=False, num_workers=4)
num_classes  = len(full_dataset.classes)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model with trained weights
model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model = model.to(device)

print(f"Evaluating on {len(full_dataset)} images across {num_classes} classes...")
results, clean_acc = evaluate_robustness(model, full_loader, device)

print("\n" + "="*65)
print(f"{'Corruption':<20} | {'Accuracy':<10} | {'Error':<10} | {'Rel. Robustness'}")
print("-"*65)
print(f"{'Clean':<20} | {clean_acc:.4f}     | {1-clean_acc:.4f}     | 1.0000")
for corruption, m in results.items():
    print(f"{corruption:<20} | {m['Accuracy']:.4f}     | {m['Corruption Error']:.4f}     | {m['Relative Robustness']:.4f}")
print("="*65)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 233MB/s] 


Evaluating on 6993 images across 30 classes...


Clean Accuracy: 0.9604



Corruption           | Accuracy   | Error      | Rel. Robustness
-----------------------------------------------------------------
Clean                | 0.9604     | 0.0396     | 1.0000
Gaussian_σ=0.05      | 0.8847     | 0.1153     | 0.9212
Gaussian_σ=0.1       | 0.7436     | 0.2564     | 0.7743
Gaussian_σ=0.2       | 0.3925     | 0.6075     | 0.4087
Motion_Blur          | 0.4450     | 0.5550     | 0.4634
Brightness_Shift     | 0.9522     | 0.0478     | 0.9915


In [7]:
import torch
import timm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn

data_dir   = "/kaggle/input/datasets/mvpranay/aid-dataset/train_data"
checkpoint = "/kaggle/input/datasets/mvpranay/gnr-models/efficientnet_b0_aid.pth"  # your saved weights

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
full_loader  = DataLoader(full_dataset, batch_size=32, shuffle=False, num_workers=4)
num_classes  = len(full_dataset.classes)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model with trained weights
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model.load_state_dict(torch.load(checkpoint, map_location=device))
model = model.to(device)

print(f"Evaluating on {len(full_dataset)} images across {num_classes} classes...")
results, clean_acc = evaluate_robustness(model, full_loader, device)

print("\n" + "="*65)
print(f"{'Corruption':<20} | {'Accuracy':<10} | {'Error':<10} | {'Rel. Robustness'}")
print("-"*65)
print(f"{'Clean':<20} | {clean_acc:.4f}     | {1-clean_acc:.4f}     | 1.0000")
for corruption, m in results.items():
    print(f"{corruption:<20} | {m['Accuracy']:.4f}     | {m['Corruption Error']:.4f}     | {m['Relative Robustness']:.4f}")
print("="*65)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 86.5MB/s]


Evaluating on 6993 images across 30 classes...


Clean Accuracy: 0.8712



Corruption           | Accuracy   | Error      | Rel. Robustness
-----------------------------------------------------------------
Clean                | 0.8712     | 0.1288     | 1.0000
Gaussian_σ=0.05      | 0.1776     | 0.8224     | 0.2039
Gaussian_σ=0.1       | 0.0159     | 0.9841     | 0.0182
Gaussian_σ=0.2       | 0.0164     | 0.9836     | 0.0189
Motion_Blur          | 0.2345     | 0.7655     | 0.2692
Brightness_Shift     | 0.8374     | 0.1626     | 0.9613
